# D11 · Clasificación y clustering — Solución

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 12 · Soluciones de referencia.


## Dos preguntas, dos tipos de aprendizaje

- **Supervisado (clasificación):** *"Tengo compras con su tamaño de proveedor conocido. ¿Puede el modelo aprender a deducir el tamaño del proveedor a partir de la cantidad y monto de la orden, para predecir el de una compra nueva?"* Hay respuestas correctas (`y`) con las que aprender.
- **No supervisado (clustering):** *"Tengo compras públicas de alimentos pero no sé cómo dividirlas en perfiles de compra. ¿Puede el algoritmo agrupar las compras más parecidas entre sí sin que yo le diga qué buscar?"* No hay respuestas correctas; el algoritmo busca similitudes numéricas en las features.


In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("compras_ml.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/B4-clasificacion-y-clustering/compras_ml.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "compras_ml.csv")
        print("compras_ml.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar compras_ml.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

df = pd.read_csv("compras_ml.csv")
print(f"{len(df)} compras cargadas. Proporción de proveedor por tamaño:")
print(df["tamano_proveedor"].value_counts())
df.head()


## 1. Clasificación: predecir una categoría

Es casi todo lo que ya sabes, con un cambio: el objetivo `y` ahora es **texto** (una categoría),
no un número. El flujo es idéntico —separar prueba, `fit`, `predict`— y hasta usaremos un
**árbol**, igual que en D10, pero su versión clasificadora: `DecisionTreeClassifier`.

> 🧠 **Analogía.** En lugar de estimar la duración de un trámite en días, decides si
> califica como "Trámite Prioritario" o "Trámite Ordinario". Hay clases fijas.

En clasificación desbalanceada, es importante usar `stratify=y` al dividir para asegurar
que el conjunto de entrenamiento y el de prueba tengan la **misma proporción** de clases que el original.


### ✍️ Ejercicio 1 — Preparar la clasificación
Define `X` con las columnas `cantidad` y `monto_total`, e `y` con `tamano_proveedor`,
y divídelos con `train_test_split` usando `test_size=0.3`, `random_state=42` y `stratify=y`.
Completa la celda.


In [ ]:
X = df[["cantidad", "monto_total"]]
y = df["tamano_proveedor"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test))


In [ ]:
# ── Celda de chequeo · Ejercicio 1 ───────────────────────────────────────────
try:
    assert list(X_train.columns) == ["cantidad", "monto_total"], \
        "X debe tener las features cantidad y monto_total."
    assert len(X_train) == 5188 and len(X_test) == 2224, \
        f"Con test_size=0.3 deben quedar 5188 y 2224; tienes {len(X_train)} y {len(X_test)}."
    assert set(y_train.unique()) == {"Micro", "Pequeña", "Mediana", "Grande"}, \
        "El objetivo y debe contener las cuatro clases: Micro, Pequeña, Mediana, Grande."
    print("✅ Correcto: dividiste los datos preservando la proporcion de las clases.")
except Exception as e:
    print("❌ Aun no. Revisa tu seleccion de features/objetivo y division, y vuelve a intentar.")
    print("   Pista:", e)


### ✍️ Ejercicio 2 — Entrenar y medir la exactitud
La métrica más simple para clasificación es la **exactitud** (*accuracy*): el **porcentaje de
casos bien clasificados**. Entrena `clasificador`, predice sobre `X_test` y calcula la exactitud
con `accuracy_score(y_test, y_pred)`, guardándola en `exactitud`. Completa la celda.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

clasificador = DecisionTreeClassifier(max_depth=3, random_state=42)
clasificador.fit(X_train, y_train)
y_pred = clasificador.predict(X_test)
exactitud = accuracy_score(y_test, y_pred)

print(f"Exactitud (accuracy) en prueba: {exactitud:.2f}")


In [ ]:
# ── Celda de chequeo · Ejercicio 2 ───────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
try:
    assert hasattr(clasificador, "tree_"), "Aun no entrenaste el clasificador: falta .fit(X_train, y_train)."
    ref = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
    acc_ref = accuracy_score(y_test, ref.predict(X_test))
    assert abs(float(exactitud) - acc_ref) < 0.01, f"Tu exactitud deberia ser ~{acc_ref:.2f}."
    print(f"✅ Correcto. El clasificador logro una exactitud de {exactitud:.2f} (un {exactitud*100:.1f}% de aciertos).")
except Exception as e:
    print("❌ Aun no. Revisa tu entrenamiento y calculo de exactitud.")
    print("   Pista:", e)


### La matriz de confusión
La exactitud resume todo en un número, pero no dice **en qué** se equivoca el modelo. La **matriz
de confusión** lo muestra: una tabla que cruza la categoría **real** (filas) con la **predicha**
(columnas). Los aciertos quedan en la **diagonal**; los errores, fuera de ella.

> 🧠 **Por qué importa en el Estado.** No es lo mismo confundir "riesgo bajo" con "riesgo medio"
> que con "riesgo alto". La matriz de confusión te muestra **qué tipo** de error comete el modelo,
> que muchas veces importa más que cuántos.


In [ ]:
# (ilustracion) Asi se lee una MATRIZ DE CONFUSION (sobre el conjunto de prueba).
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

Xi = df[["cantidad", "monto_total"]]; yi = df["tamano_proveedor"]
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=42, stratify=yi)
ci = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xi_tr, yi_tr)
y_pred_te = ci.predict(Xi_te)

# Dibujar la matriz
cm = confusion_matrix(yi_te, y_pred_te, labels=["Micro", "Pequeña", "Mediana", "Grande"])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Micro", "Pequeña", "Mediana", "Grande"],
            yticklabels=["Micro", "Pequeña", "Mediana", "Grande"])
plt.ylabel("Real")
plt.xlabel("Predicho")
plt.title("Matriz de Confusión en Prueba")
plt.show()


## 2. Clustering: descubrir grupos sin etiquetas

Cambiamos de mundo. Ahora **no** le decimos al modelo ningún tamaño de proveedor; solo le damos la cantidad y el monto de
cada compra y le pedimos que forme **grupos de compras parecidas**. Eso es **K-Means**: tú eliges
cuántos grupos quieres (*k*) y el algoritmo reparte las compras. Como cantidad y monto tienen escalas muy distintas,
es fundamental **escalar los datos** antes usando `StandardScaler`.

> 🧠 **Analogía.** Imagina que te entregan miles de expedientes de licitación sin clasificar
> y los separas en tres pilas en tu escritorio: los expedientes "delgados/baratos", los "medianos"
> y los "gigantes/complejos", simplemente por su volumen físico y costo. Eso es clustering.


### ✍️ Ejercicio 3 — Agrupar las compras con K-Means
Escala `cantidad` y `monto_total` con `StandardScaler` y agrupa las compras en **3**
clusters con `KMeans(n_clusters=3, random_state=42, n_init=10)`, guardando la etiqueta de cada
registro en `df["cluster"]`. Completa la celda.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X_compras = df[["cantidad", "monto_total"]]
X_escalado = StandardScaler().fit_transform(X_compras)
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(X_escalado)

print("Nuevos clusters asignados.")


In [ ]:
# ── Celda de chequeo · Ejercicio 3 ───────────────────────────────────────────
try:
    assert "cluster" in df.columns, "Falta crear la columna df['cluster'] con KMeans."
    assert len(df["cluster"]) == 7412, "Debe haber un cluster por cada compra."
    assert df["cluster"].nunique() == 3, \
        f"KMeans con n_clusters=3 debe producir 3 grupos distintos; hay {df['cluster'].nunique()}."
    print("✅ Correcto: escalaste los datos y asignaste cada compra a uno de los 3 clusters.")
except Exception as e:
    print("❌ Aun no. Revisa el escalamiento y la creacion de los clusters.")
    print("   Pista:", e)


In [ ]:
# (ilustracion) Las compras coloreadas por el CLUSTER que descubrio el algoritmo.
plt.figure(figsize=(7.5, 4.5))
plt.scatter(df["cantidad"], df["monto_total"], c=df["cluster"], cmap="viridis", s=15, alpha=0.5)
plt.xlabel("Cantidad de artículos")
plt.ylabel("Monto total de la compra (CLP)")
plt.title("Visualización de los 3 Clusters descubiertos por KMeans")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### ✍️ Ejercicio 4 — ¿Los grupos coinciden con el tamaño del proveedor?
Ahora comparamos lo que **descubrió** el algoritmo con la realidad. Arma una **tabla cruzada**
entre `tamano_proveedor` (real) y `cluster` (descubierto) con `pd.crosstab(df["tamano_proveedor"], df["cluster"])`,
guardándola en `tabla`. Completa la celda.


In [ ]:
import pandas as pd

tabla = pd.crosstab(df["tamano_proveedor"], df["cluster"])
print(tabla)


In [ ]:
# ── Celda de chequeo · Ejercicio 4 ───────────────────────────────────────────
try:
    assert tabla is not None, "Aun no creaste la tabla cruzada con pd.crosstab."
    assert int(tabla.values.sum()) == 7412, "La tabla debe contar las 7412 compras en total."
    assert set(tabla.index) == {"Micro", "Pequeña", "Mediana", "Grande"}, "Las filas deben ser los cuatro tamaños de proveedor."
    print("✅ Correcto. Creaste la tabla cruzada para comparar la clasificacion real con el cluster descubierto.")
except Exception as e:
    print("❌ Aun no. Revisa tu tabla pd.crosstab.")
    print("   Pista:", e)


## 3. Cierre

Ampliaste tu repertorio a los dos grandes tipos de aprendizaje:

1. **Clasificación** (supervisada): predecir una **categoría** con el mismo `fit` / `predict`,
   evaluando con **exactitud** y leyendo la **matriz de confusión** para ver *en qué* se equivoca.
2. **Clustering** (no supervisado): descubrir **patrones y grupos ocultos** sin etiquetas de antemano,
   escalando siempre antes para evitar distorsiones de magnitud y analizando con **tablas cruzadas**.

En **D12 · Pipelines reproducibles** daremos el paso definitivo hacia la madurez técnica:
empaquetarás la limpieza de datos, el escalamiento y el entrenamiento del modelo en un solo
"bloque" de producción automatizado de scikit-learn.

### Mini-glosario
- **Clasificación:** aprendizaje supervisado para predecir etiquetas discretas (categorías).
- **Exactitud (*accuracy*):** porcentaje de aciertos totales del clasificador.
- **Matriz de confusión:** tabla que muestra aciertos y confusiones por cada clase.
- **Aprendizaje no supervisado:** modelar datos sin conocer las respuestas correctas.
- **Clustering:** agrupar registros por similitud matemática.
- **Escalamiento (`StandardScaler`):** transformar variables numéricas para que tengan media 0 y varianza 1.
- **Tabla cruzada (`pd.crosstab`):** tabla de frecuencias que cruza dos variables.

---
*Fuente de datos: Dirección de Compras y Contratación Pública (ChileCompra / MercadoPúblico).*
*Contenido bajo licencia CC BY 4.0 · Formación Pública.*
